<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/Memory_types.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [83]:
!pip install -q openai

In [84]:
from pathlib import Path
from pprint import pprint
from pydantic import BaseModel
from typing import List

def print_response(response):
  print(f"Response id: {response.id}")
  print(f"Status: {response.status}")
  print(f"Input tokens: {response.usage.input_tokens} ({response.usage.input_tokens_details.cached_tokens} cached) | Output tokens: {response.usage.output_tokens}")
  pprint(response.output)

  print()

  print(f"{'=' * 20} [Text] {'=' * 20}")
  print(response.output_text)


In [113]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get('OPENROUTER_API_KEY')
client = OpenAI(api_key = api_key, base_url = "https://openrouter.ai/api/v1")

## Contextual memory
  1. Generate python class
  2. Generate instances

In [86]:
generate_class_prompt = "Generate a simple Python class that represents a person."
generate_instances_prompt = "Generate three isinstances of the person class that represent a family - mother, father and their child."

In [87]:
generate_class_response = client.responses.create(
    model = "gpt-5-nano",
    input = generate_class_prompt
)

In [88]:
print_response(generate_class_response)

Response id: resp_075d4bc56598cbaa006a48d34e91e881a3a6037deb6121f3c8
Status: completed
Input tokens: 16 (0 cached) | Output tokens: 1413
[ResponseReasoningItem(id='rs_075d4bc56598cbaa006a48d34efaec81a394b0e998c73961c9', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseOutputMessage(id='msg_075d4bc56598cbaa006a48d35a8d2c81a38da77d876ee2f031', content=[ResponseOutputText(annotations=[], text='Here\'s a simple Python class representing a person:\n\n```python\nfrom typing import Optional\n\nclass Person:\n    """A simple class representing a person with a name, age, and optional email."""\n\n    def __init__(self, name: str, age: int, email: Optional[str] = None) -> None:\n        self.name = name\n        self.age = age\n        self.email = email\n\n    def birthday(self) -> None:\n        """Increment age by 1 to simulate a birthday."""\n        self.age += 1\n\n    def __str__(self) -> str:\n        email_str = f", email={self.email}" if self.ema

In [89]:
generate_instances_response = client.responses.create(
    model = "gpt-5-nano",
    input = [
        { "role": "user", "content": generate_class_prompt },
        *generate_class_response.output,
        {"role": "user", "content": generate_instances_prompt}
    ]
)

In [90]:
print_response(generate_instances_response)

Response id: resp_075d4bc56598cbaa006a48d35d53a881a39cc569c2b8ca4139
Status: completed
Input tokens: 304 (0 cached) | Output tokens: 924
[ResponseReasoningItem(id='rs_075d4bc56598cbaa006a48d35d994481a3bd382b44ead340d9', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseOutputMessage(id='msg_075d4bc56598cbaa006a48d362225c81a38cd3da2bd582e2ab', content=[ResponseOutputText(annotations=[], text='```python\n# Assuming the Person class is already defined as in the previous example\n\n# Create three Person instances representing a family\nmother = Person("Emma", 38, "emma@example.com")\nfather = Person("Liam", 40, "liam@example.com")\nchild = Person("Noah", 10)\n\n# Optional: group them in a collection\nfamily = {"mother": mother, "father": father, "child": child}\n\n# Example: print each family member\nfor role, person in family.items():\n    print(f"{role.capitalize()}: {person}")\n```', type='output_text', logprobs=[])], role='assistant', status='comp

## Persistent memory

In [91]:
class MemorableResponse(BaseModel):
  response: str
  facts_to_remember: List[str]

PATH_TO_FACTS = Path("/content/factss.txt")

def save_facts(response: MemorableResponse) -> None:
    with PATH_TO_FACTS.open(mode="a") as file:
        file.writelines(f"- {fact}\n" for fact in response.facts_to_remember)

def read_facts() -> List[str]:
    with PATH_TO_FACTS.open(mode="r") as file:
      return file.readlines()

In [92]:
# This is a typical example for CAG(Cache Augmented Generation)
# We take all the known external information and provide it to the AI model

def ask(question: str, history = []):
  known_facts = read_facts()
  answer_response = client.responses.parse(
      model = "gpt-5-nano",
      input = [
          {"role": "developer", "content": "You are a helpful AI assistant that specializes in day-to-day interactions. Keep track of various interesting facts about the user that you may use in future to provice better responses and be even more helpful! If a fact is already known, there is no need to include it in the output."},
          *history,
          {"role": "assistant", "content": f"Facts I know about the user: {';'.join(known_facts)}"},
          {"role": "user", "content": question}
      ],
      text_format = MemorableResponse
  )

  save_facts(answer_response.output_parsed)

  return answer_response

In [93]:
greeting_promt = "Hello, AI. My name is Nikol. How do you feel today?"
ask_about_age_prompt = "I feel fine. The weather is sunny. How old are you? I am 24-years-old."
where_are_you_from_prompt = "And what about your location? I live in Bulgaria."

In [94]:
greeting_response = ask(greeting_promt)

In [95]:
print_response(greeting_response)

Response id: resp_0573f5e3205cb24e006a48d3633f388192ab8f4f95533ca269
Status: completed
Input tokens: 153 (0 cached) | Output tokens: 1072
[ResponseReasoningItem(id='rs_0573f5e3205cb24e006a48d3638bd48192bf780847034d14c1', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ParsedResponseOutputMessage[TypeVar](id='msg_0573f5e3205cb24e006a48d36979e881929ed18ad47ee85623', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"response":"Nice to meet you, Nikol! I don’t have feelings, but I’m here and ready to help. What would you like to work on today?","facts_to_remember":["User\'s name: Nikol"]}', type='output_text', logprobs=[], parsed=MemorableResponse(response='Nice to meet you, Nikol! I don’t have feelings, but I’m here and ready to help. What would you like to work on today?', facts_to_remember=["User's name: Nikol"]))], role='assistant', status='completed', type='message', phase=None)]

==================== [Text] ====================
{"resp

In [96]:
ask_about_age_response = ask(ask_about_age_prompt, history = [{"role": "user", "content": greeting_promt}, *greeting_response.output])

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MemorableResponse(respons...=["User's name: Nikol"]), input_type=MemorableResponse])
  PydanticSerializationUnexpectedValue(Expected `ResponseOutputRefusal` - serialized value may not be as expected [field_name='content', input_value=ParsedResponseOutputText[...["User's name: Nikol"])), input_type=ParsedResponseOutputText[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MemorableResponse(respons...=["User's name: Nikol"]), input_type=MemorableResponse])
  return self.__pydantic_serializer__.to_python(


In [97]:
print_response(ask_about_age_response)

Response id: resp_0573f5e3205cb24e006a48d36a7ef48192b960dd6d35ceccf5
Status: completed
Input tokens: 240 (0 cached) | Output tokens: 1395
[ResponseReasoningItem(id='rs_0573f5e3205cb24e006a48d36b31f0819287da1ee5ff36de10', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ParsedResponseOutputMessage[TypeVar](id='msg_0573f5e3205cb24e006a48d3743a3c8192853d23c389726047', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"response":"Nice to chat with you again, Nikol. I don’t have an age or birthday like people do, but I’m here to help anytime. It’s great that you’re 24! Do you have any plans for today, or would you like ideas for sunny-day activities? If you tell me your city, I can fetch a local forecast.","facts_to_remember":["User\'s age: 24"]}', type='output_text', logprobs=[], parsed=MemorableResponse(response='Nice to chat with you again, Nikol. I don’t have an age or birthday like people do, but I’m here to help anytime. It’s great that 

In [98]:
where_are_you_from_response= ask(
    where_are_you_from_prompt,
    history = [
        {"role":"user", "content": greeting_promt},
        *greeting_response.output,
        {"role":"user", "content": ask_about_age_prompt},
        *ask_about_age_response.output
    ])

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MemorableResponse(respons...mber=["User's age: 24"]), input_type=MemorableResponse])
  PydanticSerializationUnexpectedValue(Expected `ResponseOutputRefusal` - serialized value may not be as expected [field_name='content', input_value=ParsedResponseOutputText[...ber=["User's age: 24"])), input_type=ParsedResponseOutputText[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MemorableResponse(respons...mber=["User's age: 24"]), input_type=MemorableResponse])
  return self.__pydantic_serializer__.to_python(


In [99]:
print_response(where_are_you_from_response)

Response id: resp_0573f5e3205cb24e006a48d37554f481929086f4c1626e4dce
Status: completed
Input tokens: 358 (0 cached) | Output tokens: 1278
[ResponseReasoningItem(id='rs_0573f5e3205cb24e006a48d375ce188192bfab2159bfbbdff0', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ParsedResponseOutputMessage[TypeVar](id='msg_0573f5e3205cb24e006a48d37d6b2481928aa7fb1633782e45', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"response":"I don’t have a fixed physical location—I operate in the cloud, so I’m accessible from anywhere, including Bulgaria. Since you’re in Bulgaria, I can tailor information to your region (weather, time, local news, events). If you tell me a Bulgarian city, I can fetch the current weather and forecast for you. Would you like me to check the weather for a specific city (e.g., Sofia, Plovdiv, Varna)?","facts_to_remember":["User\'s location: Bulgaria"]}', type='output_text', logprobs=[], parsed=MemorableResponse(response='I d

In [100]:
# on the next day

final_response = ask("Hello, AI! What are you doing today?")

In [101]:
print_response(final_response)

Response id: resp_097129e072a674d0006a48d37e4e4881a2b6ffbab220c52b65
Status: completed
Input tokens: 169 (0 cached) | Output tokens: 1030
[ResponseReasoningItem(id='rs_097129e072a674d0006a48d37ed53c81a2a9b2b6ddabc52361', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ParsedResponseOutputMessage[TypeVar](id='msg_097129e072a674d0006a48d384f8f081a295f63f92821acd88', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{"response":"Hi Nikol! I’m here and ready to help with whatever you need today—answer questions, brainstorm ideas, plan a trip, practice Bulgarian, code, study, you name it. What would you like to start with?","facts_to_remember":[]}', type='output_text', logprobs=[], parsed=MemorableResponse(response='Hi Nikol! I’m here and ready to help with whatever you need today—answer questions, brainstorm ideas, plan a trip, practice Bulgarian, code, study, you name it. What would you like to start with?', facts_to_remember=[]))], role='as

## Semantic memory

In [103]:
tcp_ip_response = client.responses.create(
    model = "gpt-5-nano",
    input = "Explain how TCP/IP works. Use no more than 5 sentences."
)

In [105]:
print_response(tcp_ip_response)

Response id: resp_09d09afc4aa763f3006a48d46cc348819e9d77f6601ee09964
Status: completed
Input tokens: 20 (0 cached) | Output tokens: 918
[ResponseReasoningItem(id='rs_09d09afc4aa763f3006a48d46d02ec819ea9df6ec8805c637d', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseOutputMessage(id='msg_09d09afc4aa763f3006a48d4736634819eba651fe1b44fc8a6', content=[ResponseOutputText(annotations=[], text='TCP/IP is a layered set of protocols that lets devices communicate over networks, with four main layers: application, transport (TCP/UDP), internet (IP), and link.  \nIP handles addressing and routing, placing packets with IP addresses and forwarding them through routers toward the destination.  \nTCP provides reliable, connection-oriented delivery, using a three-way handshake to establish a session, sequencing and acknowledging data, and applying flow and congestion control.  \nData from applications is divided into segments (TCP) or datagrams (UDP), wrapped w

In [106]:
world_history_response = client.responses.create(
    model="gpt-5-nano",
    input="In no more than 10 sentences, describe the history of the world up until now."
)

In [107]:
print_response(world_history_response)

Response id: resp_0e7028cddf2d9599006a48d4d3c99881929fbca861c71c6a9d
Status: completed
Input tokens: 24 (0 cached) | Output tokens: 1901
[ResponseReasoningItem(id='rs_0e7028cddf2d9599006a48d4d4049081929f1cc04b697df591', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseOutputMessage(id='msg_0e7028cddf2d9599006a48d4e04be88192b6a1c17ab21bb2cd', content=[ResponseOutputText(annotations=[], text='The universe began about 13.8 billion years ago in the Big Bang, creating space, time, matter and energy. \nOur solar system formed about 4.6 billion years ago, and Earth emerged around 4.5 billion years ago with oceans and a life-sustaining atmosphere. \nLife began early on Earth as simple single-celled organisms, evolving into more complex forms over billions of years. \nThe Cambrian explosion about 541 million years ago produced a great diversity of life, followed by the movement of life onto land, including plants and later animals; dinosaurs dominated for

In [111]:
actual_news_response = client.responses.create(
    model="gpt-5-nano",
    input="List the most recent and important news from the past week."
)

In [112]:
print_response(actual_news_response)

Response id: resp_02e1ebf6e97b2c6f006a48d635331481a3b141082523ff9bd5
Status: completed
Input tokens: 18 (0 cached) | Output tokens: 1216
[ResponseReasoningItem(id='rs_02e1ebf6e97b2c6f006a48d635740881a3b7e4ebac008947aa', summary=[], type='reasoning', content=[], encrypted_content=None, status=None),
 ResponseOutputMessage(id='msg_02e1ebf6e97b2c6f006a48d63be6f081a39835b462f773e8bb', content=[ResponseOutputText(annotations=[], text='I can do that. I just need a couple of preferences to fetch and present the most relevant items.\n\nPlease tell me:\n- Region or scope: global, or a specific region (e.g., United States, Europe, Asia, etc.)\n- Topics to prioritize: politics, economy/markets, technology, science, health, climate, international conflicts, culture, etc. Any you want to exclude?\n- How many items: top 5, 10, or another number?\n- Source preference: should I pull from major outlets (Reuters, AP, BBC, NYTimes, Bloomberg, DW, etc.) or include a broader mix?\n\nIf you want, I can also

## Working Memory

In [115]:
quadratic_question_solving_response = client.responses.create(
    model="google/gemini-2.5-flash-lite",
    input = "How to solve the quadreatic equation 2*x^2-3*x+5=-1. What is the general solution plan?",
    reasoning = { "effort": "high"}
)

In [116]:
print_response(quadratic_question_solving_response)

Response id: gen-1783159344-rEcgvyO69HdsFnvrThhB
Status: completed
Input tokens: 30 (0 cached) | Output tokens: 2704
[ResponseReasoningItem(id='rs_tmp_bon5dz3o0hq', summary=[], type='reasoning', content=[Content(text="**Solving Quadratic Equation**\n\nI'm currently focused on tackling this quadratic equation. My initial step involves rearranging it into the standard form $ax^2 + bx + c = 0$ for easier analysis.\n\n**Analyzing Coefficients**\n\nI've successfully rearranged the equation into the standard quadratic form, $ax^2 + bx + c = 0$. My current focus is on identifying the specific values for $a$, $b$, and $c$ to prepare for the next steps in the solution process.\n\n**Calculating Discriminant**\n\nI'm now calculating the discriminant, $b^2 - 4ac$, using the coefficients $a=2$, $b=-3$, and $c=6$. This will tell me the nature of the roots before I proceed to the full quadratic formula application.\n\n**Finding Roots**\n\nI am currently calculating the discriminant. With $a=2$, $b=-3